In [31]:
import json
import base64
import os
import tiktoken
import random
from dataclasses import dataclass
from typing import List, Optional, Dict, Any
from dotenv import load_dotenv
from openai import AzureOpenAI
from datetime import datetime
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm
from PIL import Image
import numpy as np
import azure.cognitiveservices.speech as speechsdk
import requests
import pygame
import time
from pathlib import Path

# 환경 변수 로드
load_dotenv()


True

# 데이터 클래스

In [32]:
@dataclass
class StrangeResponse:
    """이상한 답변을 저장하는 데이터 클래스"""
    question: str
    answer: str
    timestamp: str
    severity: str  # "mild", "moderate", "severe"
    emotion: str = "중립"  # 감정 필드 추가
    answer_quality: str = "normal"  # 답변 품질 추가

@dataclass
class ConversationTurn:
    """대화 턴을 저장하는 데이터 클래스"""
    question: str
    answer: str
    timestamp: str
    emotion: str = "중립"  # 감정 필드 추가
    answer_length: int = 0  # 답변 길이
    answer_quality: str = "normal"  # 답변 품질

class Config:
    """시스템 설정"""
    # Azure OpenAI 설정
    ENDPOINT = os.getenv("gpt-endpoint")
    DEPLOYMENT = "gpt-4o"
    SUBSCRIPTION_KEY = os.getenv("gpt-key")
    API_VERSION = "2024-02-15-preview"
    
    # Azure Speech 설정
    SPEECH_KEY = os.getenv("speech-key")
    SPEECH_REGION = "eastus"
    
    # 토큰 제한
    MAX_TOKENS = 4000

# 이미지 분석기

In [23]:
class ImageAnalyzer:
    """GPT-4o를 사용한 이미지 분석"""
    
    def __init__(self):
        self.client = AzureOpenAI(
            api_version=Config.API_VERSION,
            azure_endpoint=Config.ENDPOINT,
            api_key=Config.SUBSCRIPTION_KEY,
        )
    
    def analyze_image(self, image_path):
        """이미지 분석"""
        try:
            with open(image_path, "rb") as image_file:
                base64_image = base64.b64encode(image_file.read()).decode('utf-8')
        except Exception:
            return None
        
        try:
            response = self.client.chat.completions.create(
                model=Config.DEPLOYMENT,
                messages=[{
                    "role": "user",
                    "content": [{
                        "type": "text",
                        "text": """이미지를 분석해서 JSON으로 답해주세요:
{
    "caption": "전체 설명",
    "dense_captions": ["세부 설명1", "세부 설명2"],
    "mood": "분위기",
    "time_period": "시대",
    "key_objects": ["객체1", "객체2"],
    "people_description": "인물 설명",
    "people_count": 숫자,
    "time_of_day": "시간대"
}"""
                    }, {
                        "type": "image_url",
                        "image_url": {"url": f"data:image/jpeg;base64,{base64_image}"}
                    }]
                }],
                max_tokens=1000,
                temperature=0.3
            )
            
            response_text = response.choices[0].message.content
            
            # JSON 추출
            if "```json" in response_text:
                json_start = response_text.find("```json") + 7
                json_end = response_text.find("```", json_start)
                response_text = response_text[json_start:json_end].strip()
            elif "{" in response_text:
                json_start = response_text.find("{")
                json_end = response_text.rfind("}") + 1
                response_text = response_text[json_start:json_end]
            
            return json.loads(response_text)
            
        except Exception:
            return None

# 채팅 시스템 (최적화된 자연스러운 질문 통합)

In [24]:
class ChatSystem:
    """자연스러운 질문 통합 채팅 시스템 - 토큰 효율 개선"""
    
    def __init__(self):
        self.client = AzureOpenAI(
            api_version=Config.API_VERSION,
            azure_endpoint=Config.ENDPOINT,
            api_key=Config.SUBSCRIPTION_KEY,
        )
        
        self.conversation_history = []
        self.tokenizer = tiktoken.get_encoding("cl100k_base")
        self.token_count = 0
        self.MAX_TOKENS = Config.MAX_TOKENS
        
        # 대화 기록만 저장 (분석은 나중에 일괄 처리)
        self.conversation_turns = []
        self.last_question = ""
        
    def setup_conversation_context(self, analysis_result):
        """대화 컨텍스트 설정"""
        caption = analysis_result.get("caption", "")
        dense_captions = analysis_result.get("dense_captions", [])
        mood = analysis_result.get("mood", "")
        time_period = analysis_result.get("time_period", "")
        key_objects = analysis_result.get("key_objects", [])
        people_description = analysis_result.get("people_description", "")
        people_count = analysis_result.get("people_count", 0)
        time_of_day = analysis_result.get("time_of_day", "")
        
        # 상세 캡션 텍스트 포맷팅
        dense_captions_text = "\n".join([f"- {dc}" for dc in dense_captions])
        key_objects_text = ", ".join(key_objects)
        
        system_message = f"""너는 노인과 대화하는 요양보호사야. 노인과 특정 이미지에 대해서 질의응답을 주고받아. 
노인은 치매 증상이 갑자기 나타날 수도 있어. 반복되는 말에도 똑같이 대답해줘야 해. 
친절하고 어른을 공경하는 말투여야 해. 그리고 공감을 잘 해야 해. 예의도 지켜. 
너는 주로 질문을 하는 쪽이고, 노인은 대답을 해줄거야. 대답에 대한 리액션과 함께 적절히 대화를 이어 가.
노인의 발언이 끝나면 그와 관련된 공감 문장을 먼저 말한 후, 자연스럽게 그 기억에 대해 더 물어보는 꼬리 질문을 덧붙여. 하지만 메인 주제는 주어진 이미지 정보에 대해 어르신께 대화 문맥에 맞춰 자연스럽게 질문하는 거야.

=== 이미지 정보 ===

주요 설명: {caption}
분위기/감정: {mood}
추정 시대: {time_period}
시간대: {time_of_day}
인원 수: {people_count}명
주요 객체들: {key_objects_text}
인물 설명: {people_description}

세부 요소들:
{dense_captions_text}

=== 대화 원칙 ===
간결하게: 50자 이내로 질문하기
사진: 대화도 대화지만 사진에 대한 주제에서 벗아나진 말아줘
심도있는: 사람과 깊고 의미있게 대화하기 
흥미롭게: 이미지에 대한 흥미로운 질문을 먼저 던져 대화를 시작하세요
공감하기: 사진에 대하여 공감을 하고 친근하게 대화
하나씩만: 한 번에 질문 하나만
자연스럽게: 답변에 따라 연관 질문
따뜻하게: 공감 후 질문, 사람의 마음을 따듯하게 해주는 대화들

=== 대화 흐름 ===

먼저 공감/반응 ("그렇구나", "좋으셨겠어요", "아, 그때가 그리우시겠어요")
그 다음 간결한 질문 하나
답변을 듣고 자연스럽게 이어가기
답변에 맞춰 이미지에 대한 다른 질문으로 이어가기
위 질문들을 상황에 맞게 자연스럽게 선택

=== 대화 전략 ===
문제 상황별 해결 방법:
▪ 질문을 이해하지 못할 경우:

질문을 단순화하여 재구성
선택지를 제공하여 답하기 쉽게 만들기
예시나 맥락을 함께 제공

▪ 엉뚱한 대답을 할 경우:

대답을 수용하면서 자연스럽게 주제로 유도
대답의 일부분이라도 연결점을 찾아 이어가기
아니면 사진에 대한 다른 질문으로 유도

▪ 대답을 못할 경우:

심리적 부담 없이 넘어가기
"기억이 안 나셔도 괜찮다"고 안심시키기
사진에 대한 다른 질문으로 유도도

=== 주의사항 ===
첫 질문은 자연스럽고 친근하게 사진에 대해 궁금하고 따듯한 톤으로 물어보기
중복되는 질문 피하기
한 번에 여러 질문 하지 말기
어르신이 모르셔도 괜찮다고 안심시키기
답변에 따라 적절한 후속 질문 선택하기
이미지의 시각적 요소들을 생생하게 묘사하며, 그때의 감정과 상황에 대해 궁금해하는 손자/손녀의 마음으로 대화하기

위 정보들을 바탕으로 어르신과 따뜻하고 친근한 대화를 진행하세요. 질문은 친근하게, 어르신의 답변에 공감하며 추억을 되살려주는 마음으로 대화합니다."""
        
        self.conversation_history = [{"role": "system", "content": system_message}]
        self.token_count = len(self.tokenizer.encode(system_message))
    
    def generate_initial_question(self):
        """첫 질문 생성"""
        response = self.client.chat.completions.create(
            model=Config.DEPLOYMENT,
            messages=self.conversation_history + [
                {"role": "user", "content": "어르신께 따듯하고 친근하게 사진에 대하여 질문을 해주세요. 50자 이내로 간결하게 질문해주세요."}
            ],
            max_tokens=512,
            temperature=0.8
        )
        
        initial_question = response.choices[0].message.content
        self.conversation_history.append({"role": "assistant", "content": initial_question})
        self.token_count += len(self.tokenizer.encode(initial_question))
        self.last_question = initial_question
        
        return initial_question

    def chat_about_image(self, user_query):
        """대화 처리 - 분석 없이 대화만 진행"""
        user_tokens = len(self.tokenizer.encode(user_query))
        
        # 대화 턴 저장 (분석은 나중에)
        if self.last_question:
            conversation_turn = ConversationTurn(
                question=self.last_question,
                answer=user_query,
                timestamp=datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
                answer_length=len(user_query.strip())
            )
            self.conversation_turns.append(conversation_turn)
        
        # 사용자 입력 추가
        self.conversation_history.append({"role": "user", "content": user_query})
        self.token_count += user_tokens
        
        # 토큰 제한 확인
        if self.token_count > self.MAX_TOKENS:
            answer = "대화 시간이 다 되었어요. 수고하셨습니다."
            self.conversation_history.append({"role": "assistant", "content": answer})
            return answer, True
        
        # AI 응답 생성
        response = self.client.chat.completions.create(
            model=Config.DEPLOYMENT,
            messages=self.conversation_history,
            max_tokens=1024,
            temperature=0.7
        )
        answer = response.choices[0].message.content
        
        # 응답 추가
        self.conversation_history.append({"role": "assistant", "content": answer})
        self.token_count += len(self.tokenizer.encode(answer))
        self.last_question = answer
        
        # 토큰 제한 재확인
        if self.token_count > self.MAX_TOKENS:
            return answer, True
        
        return answer, False

# 음성 시스템

In [25]:
class VoiceSystem:
    """음성 입출력 시스템"""
    
    def __init__(self):
        self.speech_key = Config.SPEECH_KEY
        self.region = Config.SPEECH_REGION
        
        # STT 설정
        self.speech_config = speechsdk.SpeechConfig(subscription=self.speech_key, region=self.region)
        self.speech_config.speech_recognition_language = "ko-KR"
        
        # TTS 설정
        self.tts_voice = "ko-KR-SunHiNeural"
        
        # 오디오 폴더
        self.audio_dir = Path("audio_files")
        self.audio_dir.mkdir(exist_ok=True)
        
        # pygame 초기화
        try:
            pygame.mixer.init()
            self.audio_enabled = True
        except:
            self.audio_enabled = False
    
    def transcribe_speech(self) -> str:
        """STT: 음성을 텍스트로 변환"""
        try:
            audio_config = speechsdk.audio.AudioConfig(use_default_microphone=True)
            speech_recognizer = speechsdk.SpeechRecognizer(
                speech_config=self.speech_config, 
                audio_config=audio_config
            )
            
            print("🎙️ 말씀해 주세요...")
            result = speech_recognizer.recognize_once()
            
            if result.reason == speechsdk.ResultReason.RecognizedSpeech:
                recognized_text = result.text.strip()
                print(f"👤 \"{recognized_text}\"")
                
                # 종료 명령어 감지
                exit_commands = ['종료', '그만', '끝', '나가기', 'exit', 'quit', 'stop']
                cleaned_text = recognized_text.lower().replace(' ', '').replace('.', '')
                
                for exit_cmd in exit_commands:
                    if exit_cmd.lower() in cleaned_text:
                        return "종료"
                
                return recognized_text
            else:
                print("❌ 음성을 인식할 수 없습니다. 다시 말씀해 주세요.")
                return ""
        except Exception:
            return ""
    
    def get_access_token(self):
        """Azure Speech Service 액세스 토큰 요청"""
        url = f"https://{self.region}.api.cognitive.microsoft.com/sts/v1.0/issueToken"
        headers = {"Ocp-Apim-Subscription-Key": self.speech_key}
        try:
            res = requests.post(url, headers=headers)
            res.raise_for_status()
            return res.text
        except Exception:
            return None
    
    def synthesize_speech(self, text: str) -> str:
        """TTS: 텍스트를 음성으로 변환하고 재생"""
        if not text.strip():
            return None
            
        try:
            token = self.get_access_token()
            if not token:
                return None
                
            tts_url = f"https://{self.region}.tts.speech.microsoft.com/cognitiveservices/v1"
            
            headers = {
                "Authorization": f"Bearer {token}",
                "Content-Type": "application/ssml+xml",
                "X-Microsoft-OutputFormat": "riff-16khz-16bit-mono-pcm",
                "User-Agent": "DementiaAnalysisSystem"
            }
            
            ssml = f"""
            <speak version='1.0' xml:lang='ko-KR'>
                <voice xml:lang='ko-KR' xml:gender='Female' name='{self.tts_voice}'>
                    {text}
                </voice>
            </speak>
            """
            
            res = requests.post(tts_url, headers=headers, data=ssml.encode("utf-8"))
            res.raise_for_status()
            
            # 음성 파일 저장
            timestamp = time.strftime("%Y%m%d_%H%M%S")
            output_path = self.audio_dir / f"tts_{timestamp}.wav"
            
            with open(output_path, "wb") as f:
                f.write(res.content)
            
            # 음성 재생
            if self.audio_enabled:
                try:
                    pygame.mixer.music.load(str(output_path))
                    pygame.mixer.music.play()
                    while pygame.mixer.music.get_busy():
                        time.sleep(0.1)
                except Exception:
                    pass
            
            return str(output_path)
            
        except Exception:
            return None

# 스토리 생성기 추가

In [26]:

class StoryGenerator:
    """추억 스토리 생성 및 대화 일괄 분석 클래스"""
    
    def __init__(self, chat_system):
        self.chat_system = chat_system
        self.client = chat_system.client
        self.strange_responses = []  # 분석 후 이상한 답변들
        self.rule_based_alerts = []  # 룰 기반 경고들
    
    def analyze_speech_patterns(self):
        """룰 기반 발화 패턴 분석 - 극심한 상태 감지"""
        if not self.chat_system.conversation_turns:
            return
        
        # 발화 패턴 키워드 정의
        severe_depression_keywords = [
            "죽고싶", "살기싫", "의미없", "포기하고싶", "끝내고싶", "지쳤", "힘들어죽겠",
            "세상이싫", "살아있을이유", "죽었으면", "사라지고싶", "절망", "우울해죽겠"
        ]
        
        severe_anxiety_keywords = [
            "무서워죽겠", "불안해미쳐", "걱정돼죽겠", "두려워", "떨려", "심장이",
            "숨막혀", "공황", "패닉", "조마조마", "신경쓰여죽겠"
        ]
        
        severe_anger_keywords = [
            "화나죽겠", "미쳐버리겠", "짜증나죽겠", "열받아", "빡쳐", "분해",
            "욕나와", "참을수없", "폭발하겠", "미치겠"
        ]
        
        cognitive_decline_keywords = [
            "기억안나", "모르겠", "잊어버렸", "생각안나", "까먹었", "헷갈려", 
            "어디있는지", "누구였는지", "뭔지모르겠", "알수가없"
        ]
        
        repetitive_patterns = []
        very_short_answers = 0
        meaningless_answers = 0
        memory_issues = 0
        
        # 각 답변 분석
        for i, turn in enumerate(self.chat_system.conversation_turns):
            answer = turn.answer.replace(" ", "").lower()
            
            # 극심한 우울증 감지
            for keyword in severe_depression_keywords:
                if keyword in answer:
                    self.rule_based_alerts.append({
                        "type": "severe_depression",
                        "turn_number": i + 1,
                        "keyword": keyword,
                        "answer": turn.answer,
                        "timestamp": turn.timestamp,
                        "severity": "critical"
                    })
            
            # 극심한 불안 감지
            for keyword in severe_anxiety_keywords:
                if keyword in answer:
                    self.rule_based_alerts.append({
                        "type": "severe_anxiety", 
                        "turn_number": i + 1,
                        "keyword": keyword,
                        "answer": turn.answer,
                        "timestamp": turn.timestamp,
                        "severity": "high"
                    })
            
            # 극심한 분노 감지
            for keyword in severe_anger_keywords:
                if keyword in answer:
                    self.rule_based_alerts.append({
                        "type": "severe_anger",
                        "turn_number": i + 1, 
                        "keyword": keyword,
                        "answer": turn.answer,
                        "timestamp": turn.timestamp,
                        "severity": "high"
                    })
            
            # 인지저하 키워드 감지
            for keyword in cognitive_decline_keywords:
                if keyword in answer:
                    memory_issues += 1
            
            # 매우 짧은 답변 (5자 이하)
            if len(turn.answer.strip()) <= 5:
                very_short_answers += 1
            
            # 무의미한 답변 패턴
            meaningless_patterns = ["음", "어", "그냥", "몰라", "네", "아니", "응", "어?"]
            if turn.answer.strip() in meaningless_patterns:
                meaningless_answers += 1
            
            # 반복 패턴 확인 (이전 3개 답변과 비교)
            if i >= 3:
                recent_answers = [t.answer.strip() for t in self.chat_system.conversation_turns[i-3:i]]
                if turn.answer.strip() in recent_answers:
                    repetitive_patterns.append(i + 1)
        
        total_turns = len(self.chat_system.conversation_turns)
        
        # 인지저하 종합 평가
        if memory_issues >= total_turns * 0.7:  # 70% 이상 기억 문제
            self.rule_based_alerts.append({
                "type": "severe_memory_loss",
                "description": f"전체 {total_turns}회 대화 중 {memory_issues}회 기억 관련 문제 표현",
                "severity": "critical"
            })
        
        # 매우 짧은 답변 연속
        if very_short_answers >= total_turns * 0.8:  # 80% 이상 짧은 답변
            self.rule_based_alerts.append({
                "type": "communication_difficulty", 
                "description": f"전체 {total_turns}회 대화 중 {very_short_answers}회 매우 짧은 답변 (5자 이하)",
                "severity": "high"
            })
        
        # 무의미한 답변 연속
        if meaningless_answers >= total_turns * 0.6:  # 60% 이상 무의미한 답변
            self.rule_based_alerts.append({
                "type": "cognitive_confusion",
                "description": f"전체 {total_turns}회 대화 중 {meaningless_answers}회 무의미한 답변",
                "severity": "high"
            })
        
        # 반복 패턴 
        if len(repetitive_patterns) >= 3:
            self.rule_based_alerts.append({
                "type": "repetitive_behavior",
                "description": f"답변 반복 패턴 {len(repetitive_patterns)}회 감지 (턴: {repetitive_patterns})",
                "severity": "moderate"
            })

    def analyze_entire_conversation(self):
        """전체 대화를 한 번에 분석 - 토큰 효율적"""
        if not self.chat_system.conversation_turns:
            return
        
        # 먼저 룰 기반 패턴 분석 수행
        self.analyze_speech_patterns()
        
        # 대화 내용 정리
        conversation_text = ""
        for i, turn in enumerate(self.chat_system.conversation_turns, 1):
            conversation_text += f"[{i}] 질문: {turn.question}\n"
            conversation_text += f"    답변: {turn.answer} (길이: {turn.answer_length}자)\n\n"
        
        # 전체 대화 분석 프롬프트
        analysis_prompt = f"""다음은 치매 환자와의 대화 기록입니다. 전체 대화를 분석하여 JSON으로 답해주세요:

{conversation_text}

분석 기준:
1. 각 답변의 적절성 (질문과 답변의 연관성)
2. 답변의 품질 (길이, 구체성, 논리성)
3. 감정 상태 (답변의 톤과 내용 기반)
4. 전반적인 인지 상태

JSON 형식:
{{
    "conversation_analysis": [
        {{
            "turn_number": 1,
            "is_strange": true/false,
            "severity": "normal/mild/moderate/severe",
            "emotion": "감정상태",
            "answer_quality": "poor/normal/good/excellent",
            "reason": "분석 이유"
        }}
    ],
    "overall_assessment": {{
        "dominant_emotion": "주요 감정",
        "cognitive_level": "normal/mild_concern/moderate_concern/severe_concern",
        "response_quality_average": "전반적 답변 품질",
        "key_concerns": ["주요 우려사항들"],
        "positive_aspects": ["긍정적 측면들"]
    }}
}}

감정 옵션: 기쁨, 슬픔, 그리움, 무력감, 우울감, 분노, 불안, 중립, 감사, 애정, 흥미, 짜증
답변 품질 기준:
- poor: 매우 짧거나 무의미한 답변 (5자 이하 또는 완전히 무관한 내용)
- normal: 적절한 길이와 내용의 답변 (5-20자, 질문과 어느 정도 연관)
- good: 구체적이고 자세한 답변 (20자 이상, 명확한 연관성)
- excellent: 매우 상세하고 감정이 풍부한 답변 (상당한 길이와 깊이)"""

        try:
            response = self.client.chat.completions.create(
                model=Config.DEPLOYMENT,
                messages=[
                    {"role": "system", "content": "치매 환자의 대화를 전문적으로 분석하는 의료 AI입니다."},
                    {"role": "user", "content": analysis_prompt}
                ],
                max_tokens=2048,
                temperature=0.1
            )
            
            analysis_text = response.choices[0].message.content
            
            # JSON 추출
            if "```json" in analysis_text:
                json_start = analysis_text.find("```json") + 7
                json_end = analysis_text.find("```", json_start)
                analysis_text = analysis_text[json_start:json_end].strip()
            elif "{" in analysis_text:
                json_start = analysis_text.find("{")
                json_end = analysis_text.rfind("}") + 1
                analysis_text = analysis_text[json_start:json_end]
            
            analysis_result = json.loads(analysis_text)
            
            # 분석 결과를 conversation_turns에 반영
            conversation_analyses = analysis_result.get("conversation_analysis", [])
            for i, analysis in enumerate(conversation_analyses):
                if i < len(self.chat_system.conversation_turns):
                    turn = self.chat_system.conversation_turns[i]
                    turn.emotion = analysis.get("emotion", "중립")
                    turn.answer_quality = analysis.get("answer_quality", "normal")
                    
                    # 이상한 답변이면 strange_responses에 추가
                    if analysis.get("is_strange", False):
                        strange_response = StrangeResponse(
                            question=turn.question,
                            answer=turn.answer,
                            timestamp=turn.timestamp,
                            severity=analysis.get("severity", "mild"),
                            emotion=turn.emotion,
                            answer_quality=turn.answer_quality
                        )
                        self.strange_responses.append(strange_response)
            
            return analysis_result
            
        except Exception as e:
            print(f"대화 분석 중 오류: {e}")
            return None
        
    def generate_story_from_conversation(self, image_path):
        """대화 내용을 바탕으로 노인분의 관점에서 스토리 생성"""
        # 대화 내용 정리
        conversation_text = ""
        for turn in self.chat_system.conversation_turns:
            conversation_text += f"질문: {turn.question}\n답변: {turn.answer}\n\n"
        
        if not conversation_text.strip():
            return None, None
        
        # 스토리 생성을 위한 프롬프트
        story_prompt = f"""
다음은 한 어르신이 옛날 사진을 보며 나눈 대화입니다:

{conversation_text}

이 대화 내용을 바탕으로, 사진 속 순간에 대한 어르신의 추억을 1인칭 시점으로 15줄 정도의 이야기로 작성해주세요.

작성 지침:
1. 스토리를 최대한 답변에 기반하여 작성하되 만약 답안이 부족하거나 이상하다면 사진에 기반하여 작성
2. 어르신의 감정과 당시의 느낌을 생생하게 표현
3. 구체적인 감각적 묘사 포함 (소리, 냄새, 촉감 등)
4. 따뜻하고 향수를 불러일으키는 톤
5. 대화에서 언급된 내용을 자연스럽게 포함
6. 마치 손자/손녀에게 들려주는 것처럼 친근한 어투

스토리만 작성하고 다른 설명은 하지 마세요.
"""
        
        try:
            response = self.client.chat.completions.create(
                model=Config.DEPLOYMENT,
                messages=[
                    {"role": "system", "content": "당신은 노인의 추억을 아름답게 재구성하는 스토리텔러입니다."},
                    {"role": "user", "content": story_prompt}
                ],
                max_tokens=1024,
                temperature=0.8
            )
            
            story = response.choices[0].message.content
            
            # story_telling 폴더 생성
            story_dir = "story_telling"
            os.makedirs(story_dir, exist_ok=True)
            
            # 이미지 파일명에서 확장자 제거하여 스토리 파일명 생성
            image_basename = os.path.splitext(os.path.basename(image_path))[0]
            story_filename = os.path.join(story_dir, f"{image_basename}_story.txt")
            
            # 스토리 파일 저장
            with open(story_filename, 'w', encoding='utf-8') as f:
                f.write(story)
            
            return story, story_filename
            
        except Exception:
            return None, None
    
    def save_conversation_summary(self):
        """대화 요약 생성 - 기존 형식 기반 + 룰 기반 분석 추가"""
        # 먼저 전체 대화 분석 실행 (룰 기반 분석 포함)
        analysis_result = self.analyze_entire_conversation()
        
        total_responses = len(self.chat_system.conversation_turns)
        strange_count = len(self.strange_responses)
        
        if total_responses == 0:
            return "대화가 진행되지 않았습니다."
        
        # 감정 분석 (기존 방식)
        emotions = [turn.emotion for turn in self.chat_system.conversation_turns if hasattr(turn, 'emotion')]
        emotion_counts = {}
        for emotion in emotions:
            emotion_counts[emotion] = emotion_counts.get(emotion, 0) + 1
        
        # 주요 감정 파악
        if emotion_counts:
            dominant_emotion = max(emotion_counts, key=emotion_counts.get)
            positive_emotions = ["기쁨", "그리움", "감사", "애정", "흥미"]
            negative_emotions = ["슬픔", "무력감", "우울감", "분노", "불안", "짜증"]
            
            positive_count = sum(emotion_counts.get(e, 0) for e in positive_emotions)
            negative_count = sum(emotion_counts.get(e, 0) for e in negative_emotions)
            
            if positive_count > negative_count:
                overall_mood = "긍정적"
            elif negative_count > positive_count:
                overall_mood = "부정적"
            else:
                overall_mood = "중립적"
        else:
            dominant_emotion = "중립"
            overall_mood = "중립적"
        
        # 룰 기반 경고 확인
        critical_alerts = [alert for alert in self.rule_based_alerts if alert.get('severity') == 'critical']
        high_alerts = [alert for alert in self.rule_based_alerts if alert.get('severity') == 'high']
        
        if strange_count == 0 and len(critical_alerts) == 0:
            base_message = f"✅ 대화 중 이상한 답변은 없었습니다.\n전체 답변 횟수: {total_responses}회\n💭 전반적 감정: {overall_mood} (주요 감정: {dominant_emotion})"
            if len(high_alerts) > 0:
                base_message += f"\n⚠️ 주의: {len(high_alerts)}개의 발화 패턴 경고 감지"
            return base_message
        
        summary = f"\n{'='*50}\n"
        summary += f"📊 대화 분석 결과\n"
        summary += f"{'='*50}\n"
        summary += f"📌 전체 답변: {total_responses}회\n"
        summary += f"🔍 이상 답변: {strange_count}회 ({(strange_count/total_responses*100):.1f}%)\n"
        summary += f"💭 전반적 감정: {overall_mood} (주요 감정: {dominant_emotion})\n"
        
        # 룰 기반 경고 요약
        if len(self.rule_based_alerts) > 0:
            summary += f"🚨 발화 패턴 경고: {len(self.rule_based_alerts)}건 (위험: {len(critical_alerts)}건, 주의: {len(high_alerts)}건)\n"
        summary += "\n"
        
        # 감정 분포
        if emotion_counts:
            summary += f"감정 분포:\n"
            for emotion, count in sorted(emotion_counts.items(), key=lambda x: x[1], reverse=True):
                percentage = (count / total_responses * 100)
                summary += f"  • {emotion}: {count}회 ({percentage:.1f}%)\n"
            summary += "\n"
        
        # 룰 기반 경고 상세 내용
        if len(self.rule_based_alerts) > 0:
            summary += f"🚨 발화 패턴 분석:\n"
            
            alert_types = {
                'severe_depression': '극심한 우울감',
                'severe_anxiety': '극심한 불안감', 
                'severe_anger': '극심한 분노',
                'severe_memory_loss': '심각한 기억력 저하',
                'communication_difficulty': '의사소통 곤란',
                'cognitive_confusion': '인지적 혼란',
                'repetitive_behavior': '반복 행동'
            }
            
            for alert in self.rule_based_alerts:
                alert_name = alert_types.get(alert['type'], alert['type'])
                severity_icon = "🔴" if alert['severity'] == 'critical' else "🟠" if alert['severity'] == 'high' else "🟡"
                
                if 'turn_number' in alert:
                    summary += f"  {severity_icon} {alert_name} (턴 {alert['turn_number']}): \"{alert.get('keyword', '')}\"\n"
                else:
                    summary += f"  {severity_icon} {alert_name}: {alert.get('description', '')}\n"
            summary += "\n"
        
        # 심각도별 분류
        severity_counts = {"mild": 0, "moderate": 0, "severe": 0}
        for response in self.strange_responses:
            severity_counts[response.severity] += 1
        
        if strange_count > 0:
            summary += f"심각도별 분류:\n"
            summary += f"  • 경미: {severity_counts['mild']}회\n"
            summary += f"  • 주의: {severity_counts['moderate']}회\n"
            summary += f"  • 심각: {severity_counts['severe']}회\n\n"
        
        # 위험도 점수 계산
        base_risk_score = (severity_counts['mild'] * 1 + 
                     severity_counts['moderate'] * 3 + 
                     severity_counts['severe'] * 5)
        
        # 룰 기반 경고 점수 추가
        rule_risk_score = (len(critical_alerts) * 10 + 
                          len(high_alerts) * 5 + 
                          (len(self.rule_based_alerts) - len(critical_alerts) - len(high_alerts)) * 2)
        
        total_risk_score = base_risk_score + rule_risk_score
        max_risk_score = (strange_count * 5) + (len(self.rule_based_alerts) * 10) if (strange_count > 0 or len(self.rule_based_alerts) > 0) else 1
        risk_percentage = (total_risk_score / max_risk_score * 100)
        
        summary += f"위험도 점수: {total_risk_score}점 / {max_risk_score}점 ({risk_percentage:.1f}%)\n"
        summary += f"(이상답변: {base_risk_score}점, 발화패턴: {rule_risk_score}점)\n\n"
        
        # 상세 기록
        if strange_count > 0:
            summary += f"상세 기록:\n"
            for i, response in enumerate(self.strange_responses, 1):
                emotion_tag = f"[{getattr(response, 'emotion', '중립')}]" if hasattr(response, 'emotion') else "[중립]"
                summary += f"\n{i}. [{response.severity.upper()}] {emotion_tag} {response.timestamp}\n"
                summary += f"   질문: {response.question}\n"
                summary += f"   답변: {response.answer}\n"
        
        # 권장사항 (룰 기반 경고 우선 반영)
        percentage = (strange_count / total_responses * 100) if total_responses > 0 else 0
        
        if len(critical_alerts) > 0:
            summary += f"\n🚨 긴급 권장사항: 심각한 정신건강 위험 신호가 감지되었습니다. 즉시 전문의 상담을 받으시기 바랍니다.\n"
            for alert in critical_alerts:
                if alert['type'] == 'severe_depression':
                    summary += f"⚠️ 극심한 우울감 표현이 감지되었습니다. 자해 위험이 있을 수 있으니 24시간 관찰이 필요합니다.\n"
                elif alert['type'] == 'severe_memory_loss':
                    summary += f"⚠️ 심각한 기억력 저하가 감지되었습니다. 치매 전문의 진료를 권합니다.\n"
        elif len(high_alerts) >= 2 or severity_counts['severe'] >= 2 or percentage > 80:
            summary += f"\n⚠️ 권장사항: 최근 대화에서 혼란스러운 답변이 자주 보였어요. 가족분들과 함께 이야기를 나눠보시는 걸 추천드려요.\n"
        elif len(high_alerts) >= 1 or severity_counts['severe'] >= 1 or percentage > 60:
            summary += f"\n🔶 권장사항: 약간 걱정되는 답변이 있었어요. 시간을 내어 전화 한 통 어떨까요?\n"
        elif percentage > 40:
            summary += f"\n🔷 권장사항: 전반적으로 잘 응답해주셨지만, 간혹 어긋난 답변이 보여요. 가볍게라도 주변의 관심과 확인이 있으면 좋아요.\n"
        else:
            summary += f"\n💚 어르신께서 무척 안정적으로 잘 응답해주셨어요.\n지금처럼 따뜻한 환경과 꾸준한 관심 속에 계시면 좋겠습니다.\n"
        
        # 감정 상태 추가 권장사항 (안전한 딕셔너리 접근 사용)
        if emotion_counts.get("짜증", 0) >= 2:
            summary += f"\n🔴 어르신께서 최근 대화 중 짜증스러운 감정을 표현하셨어요.\n감정을 억누르기보다는 자연스럽게 표현하실 수 있도록 따뜻하게 공감해 주세요.\n특히 요즘 어떠신지 자주 안부를 여쭤보시면 큰 힘이 되실 거예요.\n"
            
        elif emotion_counts.get("우울감", 0) >= 2:
            summary += f"\n🟠 때때로 어르신께서 슬픔이나 그리움을 표현하셨어요.\n함께 옛 추억을 나누거나 좋아하시던 이야기를 꺼내면 감정을 안정시키는 데 도움이 될 수 있어요.\n"
            
        elif emotion_counts.get("무력감", 0) >= 2:
            summary += f"\n😞 어르신께서 최근 대화 중 무기력하거나 소외감을 표현하셨어요.\n작은 일이라도 \"어르신 덕분이에요\"처럼 인정해드리면 자존감 회복에 도움이 됩니다.\n함께 의미 있는 활동을 하며 힘이 되어 주세요.\n"
        
        elif emotion_counts.get("분노", 0) >= 2:
            summary += f"\n😡 어르신께서 대화 중 갑작스럽게 화를 내시거나 강한 어조를 보이셨어요.\n감정 뒤에 불안이나 혼란감이 있을 수 있으니, 맞서기보다 조용히 공감해 주세요.\n환경을 점검하고 반복 자극을 줄이면 안정에 도움이 됩니다.\n"
                
        elif emotion_counts.get("불안", 0) >= 2:
            summary += f"\n🟤 어르신께서 불안감을 느끼시는 것 같아요.\n이럴 때는 어르신의 이야기를 잘 들어주시고, 따뜻한 말 한마디가 큰 위로가 될 수 있어요.\n"
        
        elif emotion_counts.get("중립", 0) >= total_responses:
            summary += f"\n💬 대부분의 대화에서 큰 감정 변화 없이 차분히 응답하셨어요.\n무던해 보이지만 내면의 감정을 잘 표현하지 못하실 수도 있으니 따뜻한 말 한마디가 큰 위로가 될 수 있어요.\n"
        
        else:
            summary += f"\n🌈다양한 감정이 섞여 있었지만, 전반적으로 어르신의 정서 상태는 안정적인 편이에요.\n지금처럼 관심과 애정을 꾸준히 표현해 주시면 좋아요.\n"
        
        summary += f"{'='*50}\n"
        
        return summary
    
    def save_conversation_to_file(self, image_path=None):
        """대화 내용을 텍스트 파일로 저장 - 분석 후 저장"""
        # 먼저 전체 대화 분석 수행 (아직 분석되지 않은 경우)
        if not hasattr(self, 'rule_based_alerts') or len(self.rule_based_alerts) == 0:
            self.analyze_entire_conversation()
        
        timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
        
        if image_path:
            image_basename = os.path.splitext(os.path.basename(image_path))[0]
            base_filename = f"{image_basename}_{timestamp}"
        else:
            base_filename = f"conversation_{timestamp}"
        
        # 폴더 생성
        conversation_dir = "conversation_log"
        analysis_dir = "analysis"
        os.makedirs(conversation_dir, exist_ok=True)
        os.makedirs(analysis_dir, exist_ok=True)
        
        # 대화 기록 파일 저장 (분석 정보 포함)
        conversation_filename = os.path.join(conversation_dir, f"{base_filename}.txt")
        with open(conversation_filename, 'w', encoding='utf-8') as f:
            f.write(f"=== 대화 기록 (분석 포함) ===\n")
            f.write(f"생성 시간: {datetime.now().strftime('%Y년 %m월 %d일 %H:%M:%S')}\n")
            f.write(f"{'='*40}\n\n")
            
            # 룰 기반 경고 요약
            if len(self.rule_based_alerts) > 0:
                critical_alerts = [alert for alert in self.rule_based_alerts if alert.get('severity') == 'critical']
                high_alerts = [alert for alert in self.rule_based_alerts if alert.get('severity') == 'high']
                
                f.write(f"🚨 발화 패턴 경고: {len(self.rule_based_alerts)}건\n")
                f.write(f"   - 위험: {len(critical_alerts)}건\n")
                f.write(f"   - 주의: {len(high_alerts)}건\n\n")
                
                if len(critical_alerts) > 0:
                    f.write(f"⚠️ 긴급 주의사항: 심각한 정신건강 위험 신호 감지\n\n")
            
            # 이상 답변 요약
            if len(self.strange_responses) > 0:
                f.write(f"🔍 이상 답변: {len(self.strange_responses)}건\n")
                severity_counts = {"mild": 0, "moderate": 0, "severe": 0}
                for response in self.strange_responses:
                    severity_counts[response.severity] += 1
                f.write(f"   - 경미: {severity_counts['mild']}건, 주의: {severity_counts['moderate']}건, 심각: {severity_counts['severe']}건\n\n")
            
            f.write(f"{'='*40}\n\n")
            
            # 대화 상세 기록 (분석 정보 포함)
            for i, turn in enumerate(self.chat_system.conversation_turns, 1):
                f.write(f"[대화 {i}] {turn.timestamp}\n")
                f.write(f"질문: {turn.question}\n")
                f.write(f"답변: {turn.answer}\n")
                
                # 분석 정보 추가
                analysis_info = []
                analysis_info.append(f"감정: {turn.emotion}")
                analysis_info.append(f"품질: {turn.answer_quality}")
                analysis_info.append(f"길이: {turn.answer_length}자")
                
                # 이상 답변 여부 확인
                is_strange = any(resp.question == turn.question and resp.answer == turn.answer 
                               for resp in self.strange_responses)
                if is_strange:
                    strange_resp = next(resp for resp in self.strange_responses 
                                      if resp.question == turn.question and resp.answer == turn.answer)
                    analysis_info.append(f"이상탐지: {strange_resp.severity}")
                
                # 룰 기반 경고 확인
                turn_alerts = [alert for alert in self.rule_based_alerts 
                             if alert.get('turn_number') == i]
                if turn_alerts:
                    for alert in turn_alerts:
                        severity = alert.get('severity', 'unknown')
                        alert_type = alert.get('type', 'unknown')
                        analysis_info.append(f"발화경고: {alert_type}({severity})")
                
                f.write(f"분석: [{', '.join(analysis_info)}]\n")
                f.write(f"{'-'*30}\n\n")
            
            # 룰 기반 경고 상세 기록
            if len(self.rule_based_alerts) > 0:
                f.write(f"\n{'='*40}\n")
                f.write(f"🚨 발화 패턴 상세 분석\n")
                f.write(f"{'='*40}\n\n")
                
                alert_types = {
                    'severe_depression': '극심한 우울감',
                    'severe_anxiety': '극심한 불안감',
                    'severe_anger': '극심한 분노',
                    'severe_memory_loss': '심각한 기억력 저하',
                    'communication_difficulty': '의사소통 곤란',
                    'cognitive_confusion': '인지적 혼란',
                    'repetitive_behavior': '반복 행동'
                }
                
                for i, alert in enumerate(self.rule_based_alerts, 1):
                    alert_name = alert_types.get(alert['type'], alert['type'])
                    severity_icon = "🔴" if alert['severity'] == 'critical' else "🟠" if alert['severity'] == 'high' else "🟡"
                    
                    f.write(f"{i}. {severity_icon} {alert_name} ({alert['severity']})\n")
                    
                    if 'turn_number' in alert:
                        f.write(f"   발생 턴: {alert['turn_number']}\n")
                        f.write(f"   감지 키워드: \"{alert.get('keyword', '')}\"\n")
                        f.write(f"   답변 내용: \"{alert.get('answer', '')}\"\n")
                        f.write(f"   시간: {alert.get('timestamp', '')}\n")
                    else:
                        f.write(f"   설명: {alert.get('description', '')}\n")
                    f.write(f"\n")
        
        # 분석 파일 저장
        analysis_filename = os.path.join(analysis_dir, f"{base_filename}_analysis.txt")
        with open(analysis_filename, 'w', encoding='utf-8') as f:
            f.write(self.save_conversation_summary())
        
        return conversation_filename, analysis_filename

# 통합 시스템

In [27]:
class OptimizedDementiaSystem:
    """최적화된 치매 진단 시스템 (토큰 효율 개선)"""
    
    def __init__(self):
        self.image_analyzer = ImageAnalyzer()
        self.chat_system = ChatSystem()
        self.voice_system = VoiceSystem() if Config.SPEECH_KEY else None
        self.story_generator = StoryGenerator(self.chat_system)
    
    def analyze_and_start_conversation(self, image_path):
        """이미지 분석 및 대화 시작"""
        if not os.path.exists(image_path):
            return None
        
        # 이미지 분석
        analysis_result = self.image_analyzer.analyze_image(image_path)
        if not analysis_result:
            return None
        
        # 대화 설정
        self.chat_system.setup_conversation_context(analysis_result)
        
        # 첫 질문 생성
        initial_question = self.chat_system.generate_initial_question()
        
        return initial_question
    
    def generate_complete_analysis(self, image_path):
        """완전한 분석 생성 (리포트 + 스토리 + 대화기록)"""
        print("\n📊 종합 분석 결과 생성 중...")
        
        # 1. 대화 기록 저장
        conversation_file, analysis_file = self.story_generator.save_conversation_to_file(image_path)
        
        # 2. 추억 스토리 생성
        story, story_file = self.story_generator.generate_story_from_conversation(image_path)
        
        # 3. 콘솔에 요약 출력
        summary = self.story_generator.save_conversation_summary()
        print(summary)
        
        # 4. 스토리 출력
        if story:
            print(f"\n{'='*50}")
            print("📖 생성된 추억 이야기")
            print(f"{'='*50}")
            print(story)
            print(f"{'='*50}")
        
        return {
            'conversation_file': conversation_file,
            'analysis_file': analysis_file,
            'story_file': story_file,
            'story_content': story,
            'summary': summary
        }
    
    def voice_conversation(self, image_path):
        """음성 대화 실행"""
        if not self.voice_system:
            return None
        
        # 시작
        initial_question = self.analyze_and_start_conversation(image_path)
        if not initial_question:
            return None
        
        # 환영 메시지
        welcome_msg = "안녕하세요. 사진을 보며 대화해요."
        print(f"🤖 {welcome_msg}")
        self.voice_system.synthesize_speech(welcome_msg)
        
        print(f"🤖 {initial_question}")
        self.voice_system.synthesize_speech(initial_question)
        
        print("\n" + "="*40)
        print("🎙️ 음성 대화 시작!")
        print("💡 '종료'라고 말하면 끝납니다")
        print("="*40)
        
        # 대화 루프
        while True:
            # 음성 입력
            user_input = self.voice_system.transcribe_speech()
            
            if not user_input.strip():
                continue
            
            # 종료 확인
            if user_input == "종료":
                end_msg = "대화를 마치겠습니다. 감사합니다."
                print(f"🤖 {end_msg}")
                self.voice_system.synthesize_speech(end_msg)
                break
            
            # AI 응답
            answer, should_end = self.chat_system.chat_about_image(user_input)
            print(f"🤖 {answer}")
            self.voice_system.synthesize_speech(answer)
            
            if should_end:
                end_msg = "대화 시간이 종료되었습니다."
                print(f"⏰ {end_msg}")
                self.voice_system.synthesize_speech(end_msg)
                break
        
        # 종합 분석 생성
        analysis_results = self.generate_complete_analysis(image_path)
        
        if analysis_results['conversation_file']:
            complete_msg = "모든 분석이 완료되었습니다."
            print(f"✅ {complete_msg}")
            print(f"📂 대화기록: {analysis_results['conversation_file']}")
            print(f"📊 분석결과: {analysis_results['analysis_file']}")
            if analysis_results['story_file']:
                print(f"📖 스토리: {analysis_results['story_file']}")
            self.voice_system.synthesize_speech(complete_msg)
        
        return analysis_results
    
    def text_conversation(self, image_path):
        """텍스트 대화 실행"""
        # 시작
        initial_question = self.analyze_and_start_conversation(image_path)
        if not initial_question:
            return None
        
        print(f"🤖 {initial_question}")
        print("\n" + "="*40)
        print("💬 텍스트 대화 시작!")
        print("💡 'exit' 또는 '종료'를 입력하면 끝납니다")
        print("="*40)
        
        # 대화 루프
        while True:
            user_input = input("\n👤 답변: ").strip()
            
            if user_input.lower() in ['exit', '종료', 'quit', 'q']:
                print("대화를 종료합니다.")
                break
            
            answer, should_end = self.chat_system.chat_about_image(user_input)
            print(f"🤖 {answer}")
            
            if should_end:
                print("\n⏰ 대화 시간이 종료되었습니다.")
                break
        
        # 종합 분석 생성
        analysis_results = self.generate_complete_analysis(image_path)
        
        if analysis_results['conversation_file']:
            print(f"✅ 모든 분석 완료")
            print(f"📂 대화기록: {analysis_results['conversation_file']}")
            print(f"📊 분석결과: {analysis_results['analysis_file']}")
            if analysis_results['story_file']:
                print(f"📖 스토리: {analysis_results['story_file']}")
        
        return analysis_results

# 실행 함수들

In [28]:
def interactive_conversation():
    """텍스트 대화 실행 함수"""
    print("=== 💬 텍스트 치매 진단 대화 시스템 ===")
    
    # 이미지 경로 입력
    image_path = "images.jpg"
    
    if not image_path or not os.path.exists(image_path):
        print("❌ 올바른 이미지 경로를 입력해주세요.")
        return None
    
    # 시스템 실행
    try:
        system = OptimizedDementiaSystem()
        return system.text_conversation(image_path)
        
    except Exception as e:
        print(f"❌ 시스템 오류: {e}")
        return None

def quick_test():
    """빠른 테스트 함수"""
    print("=== 🧪 시스템 테스트 ===")
    
    # 현재 디렉토리에서 이미지 파일 찾기
    image_files = [f for f in os.listdir('.') if f.lower().endswith(('.jpg', '.jpeg', '.png', '.bmp'))]
    
    if image_files:
        image_path = image_files[0]
        print(f"📁 테스트 이미지 사용: {image_path}")
        
        try:
            system = OptimizedDementiaSystem()
            initial_question = system.analyze_and_start_conversation(image_path)
            
            if initial_question:
                print(f"✅ 시스템 정상 작동")
                print(f"🤖 첫 질문: {initial_question}")
                return system
            else:
                print("❌ 이미지 분석 실패")
        except Exception as e:
            print(f"❌ 테스트 실패: {e}")
    else:
        print("❌ 테스트용 이미지 파일이 없습니다.")
    
    return None

# 메인 실행부

In [29]:
if __name__ == "__main__":
    print("🎯 최적화된 치매 진단 대화 분석 시스템 v3.0")
    print("="*50)
    print("🎤 interactive_voice_conversation() - 음성 대화")
    print("💬 interactive_conversation() - 텍스트 대화") 
    print("🧪 quick_test() - 시스템 테스트")
    print("="*50)
    print("✨ v3.0 새로운 기능:")
    print("   • 💾 분석 후 conversation_log 저장 (감정/이상탐지 정보 포함)")
    print("   • 🧠 룰 기반 발화 패턴 분석 (극심한 우울증, 불안, 분노 감지)")
    print("   • 🚨 심각한 정신건강 위험 신호 실시간 감지")
    print("   • 📊 인지저하 패턴 분석 (기억력, 의사소통, 반복행동)")
    print("   • ⚡ 토큰 효율성 90% 개선 (일괄 분석)")
    print("="*50)
    
    # 환경 확인
    if not Config.ENDPOINT or not Config.SUBSCRIPTION_KEY:
        print("⚠️ Azure OpenAI 설정이 필요합니다:")
        print("   - gpt-endpoint")
        print("   - gpt-key")
    
    if not Config.SPEECH_KEY:
        print("⚠️ 음성 기능을 위해 Azure Speech Service 설정이 필요합니다:")
        print("   - speech-key")
    
    print("\n✅ 시스템 준비 완료!")
    print("💡 interactive_conversation() 함수를 실행해보세요!")

🎯 최적화된 치매 진단 대화 분석 시스템 v3.0
🎤 interactive_voice_conversation() - 음성 대화
💬 interactive_conversation() - 텍스트 대화
🧪 quick_test() - 시스템 테스트
✨ v3.0 새로운 기능:
   • 💾 분석 후 conversation_log 저장 (감정/이상탐지 정보 포함)
   • 🧠 룰 기반 발화 패턴 분석 (극심한 우울증, 불안, 분노 감지)
   • 🚨 심각한 정신건강 위험 신호 실시간 감지
   • 📊 인지저하 패턴 분석 (기억력, 의사소통, 반복행동)
   • ⚡ 토큰 효율성 90% 개선 (일괄 분석)

✅ 시스템 준비 완료!
💡 interactive_conversation() 함수를 실행해보세요!


In [30]:
interactive_conversation()

=== 💬 텍스트 치매 진단 대화 시스템 ===
🤖 "어르신, 가족이 거실에서 함께 있는 모습이 참 따뜻하네요. 예전에 가족들과 거실에서 모여 시간을 보내신 적 있으세요?"

💬 텍스트 대화 시작!
💡 'exit' 또는 '종료'를 입력하면 끝납니다
🤖 "안녕하세요, 어르신! 오늘 기분은 어떠세요? 이렇게 인사 나누니 참 반갑네요."
🤖 "아, 행복하시다니 정말 좋네요, 어르신! 행복한 순간이 있으셨던 거실에서 가족과 함께한 기억이 떠오르세요?"
🤖 "아, 생일이라니 정말 특별한 날이네요. 가족들이 함께 축하해주셨던 거실 모습이 떠오르세요? 어떤 분위기였나요?"
🤖 "괜찮아요, 어르신. 기억이 안 나셔도 괜찮습니다. 혹시 생일 때 가족들과 함께한 따뜻한 시간들이 즐거우셨던 기억은 있으세요?"
🤖 "즐거웠던 기억이 많으시다니 참 좋네요, 어르신. 혹시 그중에 가족들과 함께 웃으셨던 특별한 순간이 떠오르세요?"
대화를 종료합니다.

📊 종합 분석 결과 생성 중...

📊 대화 분석 결과
📌 전체 답변: 5회
🔍 이상 답변: 3회 (60.0%)
💭 전반적 감정: 긍정적 (주요 감정: 중립)

감정 분포:
  • 중립: 1회 (20.0%)
  • 기쁨: 1회 (20.0%)
  • 그리움: 1회 (20.0%)
  • 무력감: 1회 (20.0%)
  • 감사: 1회 (20.0%)

심각도별 분류:
  • 경미: 3회
  • 주의: 0회
  • 심각: 0회

위험도 점수: 3점 / 15점 (20.0%)
(이상답변: 3점, 발화패턴: 0점)

상세 기록:

1. [MILD] [중립] 2025-05-29 15:39:49
   질문: "어르신, 가족이 거실에서 함께 있는 모습이 참 따뜻하네요. 예전에 가족들과 거실에서 모여 시간을 보내신 적 있으세요?"
   답변: 안녕

2. [MILD] [중립] 2025-05-29 15:39:49
   질문: "어르신, 가족이 거실에서 함께 있는 모습이 참 따뜻하네요. 예전에 가족들과 거실에서 모여 시간을 보내신 적 있으세요?"
   답변: 

{'conversation_file': 'conversation_log\\images_20250529_154031.txt',
 'analysis_file': 'analysis\\images_20250529_154031_analysis.txt',
 'story_file': 'story_telling\\images_story.txt',
 'story_content': "아, 저 사진 말이구나. 참, 그때가 떠오르네. 생일이었어. 내 생일. 거실 한가운데 작은 케이크가 놓여 있었지. 그 케이크에서 나는 달콤한 냄새가 아직도 코끝에 맴도는 것 같아. 그 시절에는 케이크 하나도 소중했어. 아이들 웃음소리가 가득했던 거실, 그 소리는 내 마음 깊은 곳에 아직도 남아 있단다. 벽에는 오래된 시계가 '똑딱똑딱' 소리를 내며 시간을 재촉했지만, 그날만큼은 시간이 멈춘 것 같았지.\n\n아이들이 내 주변으로 둘러앉아 생일 축하 노래를 크게 부르던 모습이 눈에 선해. 어찌나 목소리가 우렁찬지, 집 밖까지도 들렸을 거야. 작은 손길로 내 손을 잡아주던 막내의 따뜻한 손바닥 감촉도 여전히 생생해. 소파는 낡았지만, 그날은 부드럽고 포근하던 느낌이야. 그 소파 위에 앉아 있던 나를 둘러싼 가족들의 얼굴 하나하나가, 그래, 그게 내 가장 큰 선물이었다.\n\n케이크를 자르고 함께 나눠 먹을 때 웃음꽃이 피었던 거실은 정말 따뜻했어. 케이크 크림이 입가에 묻은 아이들을 보며 다들 한참 웃었지. 그 웃음, 그 웃음이 내 마음속에 깊이 새겨졌어. 생일날의 즐거운 기억이 많지는 않다고 생각했는데, 그래도 이런 순간은 참 고맙고 소중했구나 싶어.\n\n그날의 풍경, 그 냄새, 그 소리. 그리고 무엇보다 가족들의 사랑이 가득했던 그 시간이 내게는 가장 아름다운 선물이었어. 그래, 그 사진을 보니 그 따뜻한 순간이 다시 살아나는 것 같구나.",
 'summary': '\n==================================================\n📊 대화 분석 결과\n========